# hj_reachability quickstart

Notebook dependencies:
- System: python3, ffmpeg (for rendering animations)
- Python: jupyter, jax, numpy, matplotlib, plotly, tqdm, hj_reachability

Example setup for a Ubuntu system (Mac users, maybe `brew` instead of `sudo apt`; Windows users, learn to love [WSL](https://docs.microsoft.com/en-us/windows/wsl/install-win10)):
```
sudo apt install ffmpeg
/usr/bin/python3 -m pip install --upgrade pip
pip install --upgrade jupyter jax[cpu] numpy matplotlib plotly tqdm hj-reachability
jupyter notebook  # from the directory of this notebook
```
Alternatively, view this notebook on [Google Colab](https://colab.research.google.com/github/StanfordASL/hj_reachability/blob/main/examples/quickstart.ipynb) and run a cell containing this command:
```
!pip install --upgrade hj-reachability
```

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import time
from tqdm.notebook import tqdm

from IPython.display import HTML
import matplotlib.animation as anim
import matplotlib.pyplot as plt
import plotly.graph_objects as go

import hj_reachability as hj

from hj_reachability.systems.doubleint import DoubleIntegrator

## Example system: `Double Integrator`

In [ ]:
# dynamics = hj.systems.Air3d()
u_max = 1.0
d_max = 1.0

u_maxes = np.linspace(0.1, 1.0, 10)
d_maxes = np.linspace(0.1, 1.0, 10)

# print("u_maxes:", u_maxes)
# print("d_maxes:", d_maxes)

grid = hj.Grid.from_lattice_parameters_and_boundary_conditions(
    hj.sets.Box(np.array([-20., -10.]), np.array([20., 10.])),
    (101, 50)
)

safety_bound = 5
values = jnp.linalg.norm(grid.states[..., :2], axis=-1) - safety_bound

solver_settings = hj.SolverSettings.with_accuracy(
    "very_high",
    hamiltonian_postprocessor=hj.solver.backwards_reachable_tube)

### `hj.step`: propagate the HJ PDE from `(time, values)` to `target_time`.

In [ ]:
time_start_sim = 0.
target_time = -2.

results = {}

total_pairs = len(u_maxes) * len(d_maxes)
print(f"Starting sweep over {total_pairs} pairs...")
total_start_time = time.perf_counter()

progress_bar = tqdm(total=total_pairs, desc="HJI Parameter Sweep")

for u in u_maxes:
    for d in d_maxes:
        # Re-instantiate dynamics with current loop parameters
        dynamics = DoubleIntegrator(u_max=u, d_max=d)
        
        # Run HJI step
        target_values = hj.step(solver_settings, dynamics, grid, time_start_sim, values, target_time)
        
        # Force JAX to complete execution before saving
        target_values.block_until_ready()
        
        results[(u, d)] = target_values
        
        progress_bar.set_postfix({"u_max": f"{u:.2f}", "d_max": f"{d:.2f}"})
        progress_bar.update(1)

progress_bar.close()

total_end_time = time.perf_counter()
total_elapsed = total_end_time - total_start_time

print("=========================================")
print(f"Sweep Completed Successfully!")
print(f"Total Computation Time: {total_elapsed:.2f} seconds")
print(f"Average Time per Pair: {total_elapsed / len(results):.4f} seconds")
print("=========================================")

### Show a matrix of sampled (u_{max}, d_{max}) pair backward reachable tubes

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

num_samples = 4
sample_indices = np.linspace(0, len(u_maxes) - 1, num=num_samples, dtype=int)
plot_u = u_maxes[sample_indices]
plot_d = d_maxes[sample_indices]

fig, axes = plt.subplots(len(plot_u), len(plot_d), figsize=(16, 14), sharex=True, sharey=True)
cf = None

for r_idx, u in enumerate(plot_u):
    for c_idx, d in enumerate(plot_d):
        ax = axes[r_idx, c_idx]
        
        # Retrieve the pre-computed backward reachable tube array from your sweep dictionary
        tube_data = results[(u, d)]
        
        # Plot the continuous value landscape background
        cf = ax.contourf(grid.coordinate_vectors[0], 
                         grid.coordinate_vectors[1], 
                         tube_data.T, 
                         levels=15, 
                         cmap='viridis', 
                         alpha=0.8)
        
        # Explicitly highlight the 0-level set boundary (the exact tube wall)
        ax.contour(grid.coordinate_vectors[0], 
                   grid.coordinate_vectors[1], 
                   tube_data.T, 
                   levels=[0], 
                   colors="red", 
                   linewidths=2.0)
        
        # Subplot labels for immediate visual tracking of parameter impact
        ax.set_title(f"$u_{{max}}$: {u:.2f} | $d_{{max}}$: {d:.2f}", fontsize=9)
        ax.grid(True, alpha=0.25)

# Exterior axis labels across the matrix margins
for ax in axes[-1, :]:
    ax.set_xlabel("Position", fontsize=11)
for ax in axes[:, 0]:
    ax.set_ylabel("Velocity", fontsize=11)

# Adjust the right margin of the subplots to leave exactly enough room for the bar
fig.subplots_adjust(right=0.86)
cbar_ax = fig.add_axes([0.89, 0.15, 0.025, 0.7])  # [left, bottom, width, height]
cbar = fig.colorbar(cf, cax=cbar_ax)
cbar.set_label('Reachability Value $V(x,t)$', rotation=270, labelpad=15, fontsize=12)

# Overlay the 0-level cutoff line directly onto the colorbar scale
cbar.ax.axhline(0, color='red', linewidth=2.5)


plt.suptitle(f"Sampled Reachable Sets\n(Safety Bound = {safety_bound})", 
             fontsize=16, y=0.96, fontweight='bold')

plt.show()

### Show a single (u_{max}, d_{max}) pair backward reachable tube

In [ ]:
time_start_sim = 0.
target_time = -2.

dynamics = DoubleIntegrator(u_max=u_max, d_max=d_max)

target_values = hj.step(solver_settings, dynamics, grid, time_start_sim, values, target_time)

plt.jet()
plt.figure(figsize=(13, 8))

plt.contourf(grid.coordinate_vectors[0], 
             grid.coordinate_vectors[1], 
             target_values.T,
             cmap='viridis')
plt.colorbar()

plt.contour(grid.coordinate_vectors[0],
            grid.coordinate_vectors[1],
            target_values.T,
            levels=[0],
            # colors="black",
            linewidths=3,
            cmap='viridis')

plt.xlabel("Position")
plt.ylabel("Velocity")
plt.title(f"Double Integrator Reachable Sets (safety bound={safety_bound}, u_max={u_max}, d_max={d_max}) at time {-target_time}")

In [ ]:
go.Figure(data=go.Isosurface(x=grid.states[..., 0].ravel(),
                             y=grid.states[..., 1].ravel(),
                             z=grid.states[..., 2].ravel(),
                             value=target_values.ravel(),
                             colorscale="jet",
                             isomin=0,
                             surface_count=1,
                             isomax=0))

### `hj.solve`: solve for `all_values` at a range of `times` (basically just iterating `hj.step`).

In [ ]:
times = np.linspace(0, -2.8, 57)
initial_values = values
all_values = hj.solve(solver_settings, dynamics, grid, times, initial_values)

In [ ]:
vmin, vmax = all_values.min(), all_values.max()
levels = np.linspace(round(vmin), round(vmax), round(vmax) - round(vmin) + 1)
fig = plt.figure(figsize=(13, 8))

def render_frame(i, colorbar=False):
    plt.clf() # Clear the current figure frame to prevent overlapping artifacts
    
    plt.contourf(grid.coordinate_vectors[0],
                 grid.coordinate_vectors[1],
                 all_values[i, :, :].T,
                 vmin=vmin,
                 vmax=vmax,
                 levels=levels)
    if colorbar:
        plt.colorbar()
        
    plt.contour(grid.coordinate_vectors[0],
                grid.coordinate_vectors[1],
                target_values.T,
                levels=[0],
                colors="black",
                linewidths=3)
    
    plt.xlabel("Position")
    plt.ylabel("Velocity")
    plt.title(f"Double Integrator Time Step: {i}")


render_frame(0, True)
animation = HTML(anim.FuncAnimation(fig, render_frame, all_values.shape[0], interval=50).to_html5_video())
plt.close(); animation

## Defining your own dynamics: `AccelerationCurvatureCar`

In [ ]:
class AccelerationCurvatureCar(hj.ControlAndDisturbanceAffineDynamics):

    def __init__(self,
                 max_acceleration=1.,
                 max_curvature=1.,
                 max_position_disturbance=0.25,
                 control_mode="min",
                 disturbance_mode="max",
                 control_space=None,
                 disturbance_space=None):
        if control_space is None:
            control_space = hj.sets.Box(jnp.array([-max_acceleration, -max_curvature]),
                                        jnp.array([max_acceleration, max_curvature]))
        if disturbance_space is None:
            disturbance_space = hj.sets.Ball(jnp.zeros(2), max_position_disturbance)
        super().__init__(control_mode, disturbance_mode, control_space, disturbance_space)

    def open_loop_dynamics(self, state, time):
        _, _, v, q = state
        return jnp.array([v * jnp.cos(q), v * jnp.sin(q), 0., 0.])

    def control_jacobian(self, state, time):
        v = state[2]
        return jnp.array([
            [0., 0.],
            [0., 0.],
            [1., 0.],
            [0., v],
        ])

    def disturbance_jacobian(self, state, time):
        return jnp.array([
            [1., 0.],
            [0., 1.],
            [0., 0.],
            [0., 0.],
        ])

In [ ]:
dynamics = AccelerationCurvatureCar()
grid = hj.Grid.from_lattice_parameters_and_boundary_conditions(hj.sets.Box(lo=np.array([-5., -5., -1., -np.pi]),
                                                                           hi=np.array([5., 5., 1., np.pi])),
                                                               (40, 40, 50, 50),
                                                               periodic_dims=3)
values = jnp.linalg.norm(grid.states[..., :2], axis=-1) - 1

solver_settings = hj.SolverSettings.with_accuracy("low")

In [ ]:
time = 0.
target_time = -2.0
target_values = hj.step(solver_settings, dynamics, grid, time, values, target_time)

In [ ]:
go.Figure(data=go.Isosurface(x=grid.states[:, :, -1, :, 0].ravel(),
                             y=grid.states[:, :, -1, :, 1].ravel(),
                             z=grid.states[:, :, -1, :, 3].ravel(),
                             value=target_values[:, :, -1, :].ravel(),
                             colorscale='jet',
                             isomin=0,
                             surface_count=1,
                             isomax=0))